# Day 3 Part 2 — Quality and BLAST in Python
## Beginner Edition · Day 3 (Friday, May 1)

**Course:** Introduction to Bioinformatics — HCT Cohort
**Instructor:** Norah AlGhamdi

---

### What you'll do
In Part 1, you read a FastQC report and ran a BLAST search on the web. Now we'll do the **same things in Python code** — small, gentle steps.

By the end you will have:

- 📊 Inspected quality scores in a real FASTQ file
- ⚡ Calculated basic quality statistics
- 🔍 Submitted a BLAST search from Python
- 📄 Read the top hit and its E-value from your code

### How to use this notebook
1. **Save your own copy:** `File → Save a copy in Drive`
2. **Run each cell with `Shift + Enter`**
3. Most cells are filled in. Just run them.
4. **✏️ Your Turn** = a small thing you change.

**No big code.** Each cell does one tiny thing. 🐢


---
## Setup — install Biopython


In [ ]:
!pip install biopython --quiet


In [ ]:
from Bio import SeqIO
from Bio.Seq import Seq
from Bio.Blast import NCBIWWW, NCBIXML

print("Ready!")


---
# Section A — Quality Control

We'll create a small FASTQ file with 5 reads, then explore its quality scores.


---
## Step A1 — Create a small FASTQ file

Normally a FASTQ comes from a sequencing machine. For this exercise, we'll write a small one ourselves so we have something to play with.


In [ ]:
fastq_text = """@read_1
ATGGATTTATCTGCTCTTCGCGTTGAAGAAGTACAAAATGTCATTAATGCTATGCAGAAA
+
IIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII
@read_2
ATCTTAGAGTGTCCCATCTGTCTGGAGTTGATCAAGGAACCTGTCTCCACAAAGTGTGAC
+
IIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII???????+++++++++!!
@read_3
CACATATTTTGCAAATTTTGCATGCTGAAACTTCTCAACCAGAAGAAAGGGCCTTCACAG
+
IIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII
@read_4
TGTCCTTTATGTAAGAATGATATAACCAAAAGGAGCCTACAAGAAAGTACGAGATTTAGT
+
IIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII
@read_5
CAACTAACAACGCTGCATGTGAACATGGTAGCAGCAATCATCATGAACGCATCAGCAGCT
+
!!!!!!!!!!!+++++++++++++???????????IIIIIIIIIIIIIIIIIIIIIIIII"""

# Save to a file
with open("tiny.fastq", "w") as f:
    f.write(fastq_text)

print("Saved tiny.fastq with 5 reads")


Notice the **quality lines** (the 4th line of each read). Different characters mean different quality scores:

- `I` = high quality (~Q40)
- `?` = medium quality (~Q30)
- `+` = lower quality (~Q10)
- `!` = very poor (~Q0)

Look at read_2 and read_5 — they have lower-quality endings than the others.


---
## Step A2 — Read the FASTQ with Biopython

One line and Biopython gives us the sequences AND the quality scores.


In [ ]:
for record in SeqIO.parse("tiny.fastq", "fastq"):
    print("Read ID:    ", record.id)
    print("Sequence:   ", record.seq)
    qualities = record.letter_annotations["phred_quality"]
    print("Qualities:  ", qualities[:10], "...")
    print("---")


**Look at that.** Biopython converted the cryptic `!+?I` characters into actual numbers — Phred scores. Much easier to work with.


---
## Step A3 — Calculate the average quality per read

A quick health check: what's the average Phred score for each read?


In [ ]:
for record in SeqIO.parse("tiny.fastq", "fastq"):
    qualities = record.letter_annotations["phred_quality"]
    avg = sum(qualities) / len(qualities)
    print(f"{record.id}: average Q = {avg:.1f}")


**Reads 1, 3, and 4** have high averages — those are clean.

**Reads 2 and 5** have lower averages — they have low-quality bases mixed in.

*This is exactly the kind of check FastQC does, but for millions of reads at once.*


---
## Step A4 — Where in the read does quality drop?

Let's check: across ALL reads, what's the average quality at each position?


In [ ]:
# Get all reads as a list
all_reads = list(SeqIO.parse("tiny.fastq", "fastq"))

# Get the length of one read
read_length = len(all_reads[0].seq)

# Calculate average quality at each position
print("Position : Average Q")
print("-" * 25)
for pos in range(0, read_length, 5):  # every 5th position
    qs = [r.letter_annotations["phred_quality"][pos] for r in all_reads]
    avg = sum(qs) / len(qs)
    print(f"  {pos:3}    : {avg:.1f}")


Notice the pattern — the average quality often **drops at the end** of the reads. This is the same pattern you saw in the bad FastQC report.


---
## Step A5 — Trim the bad ends

Let's trim away anything below Q20 from the end of each read.


In [ ]:
def trim_end(record, min_q=20):
    """Trim low-quality bases from the end of a read."""
    qualities = record.letter_annotations["phred_quality"]
    # Walk back from the end while quality is low
    end = len(qualities)
    while end > 0 and qualities[end - 1] < min_q:
        end -= 1
    return record[:end]   # return the trimmed record

for record in SeqIO.parse("tiny.fastq", "fastq"):
    trimmed = trim_end(record)
    print(f"{record.id}: {len(record.seq)} → {len(trimmed.seq)} bp after trimming")


**Reads 1, 3, 4** stay the same length — they were already clean.

**Reads 2 and 5** lose their bad endings — now they're shorter but trustworthy.

*That's quality control in 5 small steps. Same idea as FastQC + Trimmomatic — just on tiny data so you can see what's happening.*


---
# Section B — BLAST in Python

In Part 1 you BLASTed a sequence on the NCBI website. Now we do the same thing in Python.


---
## Step B1 — Pick a sequence to BLAST

We'll use the same mystery sequence from Part 1.


In [ ]:
mystery_sequence = """ATGGCCCTGTGGATGCGCCTCCTGCCCCTGCTGGCGCTGCTGGCCCTCTGGGGACCTGAC
CCAGCCGCAGCCTTTGTGAACCAACACCTGTGCGGCTCACACCTGGTGGAAGCTCTCTAC
CTAGTGTGCGGGGAACGAGGCTTCTTCTACACACCCAAGACCCGCCGGGAGGCAGAGGAC
CTGCAGGTGGGGCAGGTGGAGCTGGGCGGGGGCCCTGGTGCAGGCAGCCTGCAGCCC"""

# Clean it up — remove the line breaks
mystery_sequence = mystery_sequence.replace("\n", "")

print("Length:", len(mystery_sequence), "bases")
print("First 60:", mystery_sequence[:60])


---
## Step B2 — Submit the BLAST search

**Heads up:** this cell calls NCBI's servers. It takes **30 seconds to 2 minutes** to finish — be patient. ⏳


In [ ]:
print("Submitting to NCBI BLAST... (please wait 30-60 seconds)")

# Submit to BLAST: blastn (DNA) against the nt database
result_handle = NCBIWWW.qblast("blastn", "nt", mystery_sequence)

# Save the result to a file
with open("blast_result.xml", "w") as f:
    f.write(result_handle.read())
result_handle.close()

print("Done! Saved to blast_result.xml")


---
## Step B3 — Read the top hits

BLAST returns its results in XML format. Biopython knows how to read it.


In [ ]:
with open("blast_result.xml") as f:
    blast_record = NCBIXML.read(f)

print("Number of hits:", len(blast_record.alignments))
print()

# Show the top 3
print("Top 3 hits:")
print("=" * 70)
for alignment in blast_record.alignments[:3]:
    print("Title:   ", alignment.title[:80])
    print("Length:  ", alignment.length, "bp")
    hsp = alignment.hsps[0]
    print(f"E-value: {hsp.expect}")
    print(f"Score:   {hsp.score}")
    print("-" * 70)


**The top hit should be insulin** (the gene for the insulin protein) — same answer you got from the website.

*This is how researchers automate BLAST searches when they have many sequences to identify.*


---
## Step B4 — Just print the best match cleanly

If you only care about the top hit:


In [ ]:
top = blast_record.alignments[0]
top_hsp = top.hsps[0]

print("BEST MATCH FOUND:")
print("=" * 50)
print("Description: ", top.title[:80])
print("E-value:     ", top_hsp.expect)
print("Identity:    ", top_hsp.identities, "/", top_hsp.align_length, "bases match")
percent = (top_hsp.identities / top_hsp.align_length) * 100
print(f"% Identity:  {percent:.1f}%")


---
## Step B5 — ✏️ Your Turn — BLAST your own sequence

**Pick any short DNA sequence and BLAST it from Python.**

Replace the sequence below with one of your own (e.g., copy a piece from your favorite gene's NCBI page).


In [ ]:
# ✏️ Replace with your own sequence
my_sequence = "ATGCAGAAATCTTAGAGTGTCCCATCTGTCTGGAGTTGATCAAGGAACCTGTCTCCACAA"

print("Submitting to BLAST...")
result_handle = NCBIWWW.qblast("blastn", "nt", my_sequence)

with open("my_blast.xml", "w") as f:
    f.write(result_handle.read())
result_handle.close()

print("Done!")

# Read the top hit
with open("my_blast.xml") as f:
    record = NCBIXML.read(f)

if len(record.alignments) > 0:
    top = record.alignments[0]
    print("Top hit:", top.title[:80])
    print("E-value:", top.hsps[0].expect)
else:
    print("No hits found.")


---
## 📚 Homework — Apply what you learned

**Optional, but encouraged.** Bring answers to next session.


### Homework 1 — How many reads passed?

From `tiny.fastq`, count how many of the 5 reads have an average quality above Q30.


In [ ]:
count_passed = 0
for record in SeqIO.parse("tiny.fastq", "fastq"):
    qualities = record.letter_annotations["phred_quality"]
    avg = sum(qualities) / len(qualities)
    if avg > 30:
        count_passed += 1

print(count_passed, "out of 5 reads passed (avg Q > 30)")


### Homework 2 — BLAST a piece of BRCA1

Take the first 60 bases of BRCA1 (from yesterday's notebook) and BLAST them. **Does BLAST identify it as BRCA1?**


In [ ]:
# A piece of human BRCA1
brca1_piece = "ATGGATTTATCTGCTCTTCGCGTTGAAGAAGTACAAAATGTCATTAATGCTATGCAGAAA"

print("Submitting BRCA1 piece to BLAST...")
result_handle = NCBIWWW.qblast("blastn", "nt", brca1_piece)

with open("brca1_blast.xml", "w") as f:
    f.write(result_handle.read())
result_handle.close()

with open("brca1_blast.xml") as f:
    rec = NCBIXML.read(f)

if len(rec.alignments) > 0:
    top = rec.alignments[0]
    print("Top hit:", top.title[:80])
    print("E-value:", top.hsps[0].expect)
    print("Did BLAST find BRCA1?")


### Homework 3 — Bonus 🎁

Take a sequence from a gene OTHER than BRCA1 — pick anything from NCBI. BLAST it from Python.

Bring to the next session:
- The gene you tried
- The top BLAST hit it returned
- The E-value

*That's a complete real-world bioinformatics task — you're doing it.*


---
## 🎉 You did it!

Today you learned to do — in Python — what bioinformaticians do every day:

| Skill | Why it matters |
|---|---|
| Read a FASTQ in Python | Every sequencing dataset starts here |
| Calculate quality scores | The first quality check before any analysis |
| Trim low-quality bases | Cleaning your data before using it |
| Submit BLAST from code | Automating sequence identification |
| Parse BLAST results | Extracting answers from XML |

### What's next — tomorrow

Tomorrow we'll go deeper into **sequence alignment** (multiple sequences this time, not just one) and **variants** — the differences that make individuals unique.

**Great work today. See you tomorrow!** 👋
